## **Nugen Intelligence**
<img src="https://nugen.in/logo.png" alt="Nugen Logo" width="200"/>

Domain-aligned foundational models at industry leading speeds and zero-data retention! To learn more, visit [Nugen](https://docs.nugen.in/introduction)

### **Vision Alignmeent with the Nugen API**
 
This cookbook demonstrates how to create a Vision Alignment Project using the Nugen API. You'll learn how to upload an image dataset, automatically generate benchmark questions, train an aligned vision model, monitor training progress, and finally perform inference using the aligned model.

The notebook explains each step in a simple, sequential manner so that you can easily reproduce the complete workflow.

**Vision Model**

The vision model can interpret images and generate responses based on both images and text. This is useful in scenarios like captioning images or answering questions about them.


**Workflow**                                                                                  
The cookbook covers the following steps:

1 Upload a vision dataset (.jsonl).
2 Retrieve document details.
3 Generate benchmark questions from the uploaded dataset.
4 Create a Vision Alignment project.
5 Monitor alignment training status.
6 Run inference using the aligned vision model.

**Dataset Split**

After uploading your dataset, 15% of the uploaded data will automatically be used to generate benchmark questions for evaluating the aligned model. The remaining data is used during the alignment process.

Uploaded Dataset
      - 85% → Alignment Training
      - 15% → Benchmark Generation

**Install the required Python packages**

In [26]:
!pip install --quiet requests pandas python-dotenv

2561.32s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


**Import Required Libraries**

In [29]:
import os 
import requests
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

False

**Set up the Nugen API Client**

To read more about Nugen API and access free API keys, you can visit [Nugen Dashboard](https://nugen-platform-frontend.azurewebsites.net/dashboard)

**Configure the API Client**

Set your Nugen API key.

In [ ]:
import os

api_key = os.getenv("NUGEN_API_KEY")

# Or replace with your API key

In [ ]:
# api_key = "nugen-xxxxxxxxxxxxxxxx"

Here, we define the API base URL and your API key. Replace <--nugen api key--> with your actual key to authenticate your requests to the Nugen API. The MODEL variable specifies the model we will use for generating the routines.

In [ ]:
BASE_URL = "https://api.nugen.in"

**Step 1 - Upload the Vision Dataset**

Upload a JSONL dataset

In [ ]:
url = f"{BASE_URL}/api/v3/documents"

In [ ]:
headers = {"Authorization": f"Bearer {api_key}"}

with open("/home/abhishek/nugen-cookbook/guides/Vision_alignment/vision_dataset.jsonl", "rb") as f:
    files = {
        "files": ("vision_dataset.jsonl", f, "application/json")
    }

    data = {
        "categories": "image"
    }

    response = requests.post(
        url,
        headers=headers,
        data=data,
        files=files
    )
print(response.text)

200
{"document_ids":["01KWGZ3FYWRSFT8"]}


**Step 2 - Get status of Document uploaded**

Once the upload is complete, retrieve the document ID

This endpoint returns metadata such as the document ID and processing status.

In [ ]:
url = f"{BASE_URL}/api/v3/documents/{document_id}"

In [ ]:
headers = {"Authorization": f"Bearer {api_key}"}

response = requests.get(url, headers=headers)

print(response.text)

{"status":"READY","document_id":"doc_01KWGZ3G6N9BVVT"}


**Step 3 - Generate Benchmark Questions**

Generate evaluation questions from the uploaded dataset.

Note: Benchmark questions are generated using 15% of the uploaded dataset

In [ ]:
url = f"{BASE_URL}/api/v3/benchmark/create"

In [ ]:
payload = {
    "documents": ["doc_123"],
    "num_questions": 20
}
headers = {
    "Authorization": f"Bearer {api_key}"
}

response = requests.post(url, json=payload, headers=headers)

print(response.text)

{"benchmark_id":"benchmark_01KWGZDCKYJD0YC","status":"PROCESSING"}


**Step 4 - Get status Benchmark Generation**

Check whether benchmark generation has completed.

In [ ]:
url = f"{BASE_URL}/api/v3/benchmark/create/{benchmark_id}"

In [ ]:
headers = {
    "Authorization": f"Bearer {api_key}"
}

response = requests.get(url, headers=headers)

print(response.text)

{"benchmark_id":"benchmark_01KWGZDCKYJD0YC","benchmark_name":"generated_benchmark_01KWGZDCKYJD0YC","status":"READY","start_time":"2026-07-02T08:34:03.479673","end_time":"2026-07-02T08:34:03.961290"}


**Step 5 - Create a Vision Alignment Project**

Create an alignment project using:

-Base vision model

-Uploaded document

-Generated benchmark

-Project description

**Example base model:**

qwen2-vl-2b-instruct

In [ ]:
url = f"{BASE_URL}/api/v3/alignment-project/create"

In [ ]:
payload = {
    "name": "My Vision Alignment",
    "base_model": "qwen2-vl-2b-instruct",
    "document_ids":["doc_123"],
    "workflow_id": "workflow-abc123",
    "benchmark_id": "benchmark_123abc",
    "description": "This project aims to align the model for better vision alignment."
}

headers = {
    "Authorization": f"Bearer {api_key}"
}
response = requests.post(url, json=payload, headers=headers)

print(response.text)

{"alignment_id":"alignment_01KWGZPYERMGQBT","status":"PROCESSING"}


**Step 6 - Check Alignment Status**

Track the alignment status.

In [ ]:
url = f"{BASE_URL}/api/v3/alignment-project/status/{alignment_id}"

In [ ]:
headers = {
    "Authorization": f"Bearer {api_key}"
}
response = requests.get(url, headers=headers)

print(response.text)

{"status":"PROCESSING","data":null,"start_time":"2026-07-02T08:40:36.712847","end_time":null,"gpu_training_completed":null}


**Step 7 - Run Inference**

Once alignment completes, you'll receive an Aligned Model ID.
Use this model for inference.

In [ ]:
url = f"{BASE_URL}/api/v3/chat/completions"

In [ ]:
model_id= "Aligned model id"

In [ ]:
headers = {
    "Authorization": f"Bearer {api_key}"
}

payload = {
    "model": model_id,
    "messages": [
        {
            "role": "system",
            "content": "Tell me about my image"
        }
    ],
    "max_tokens": 100,
    "prompt_truncate_len": 123,
    "temperature": 1,
    "stream": False,
}

response = requests.post(url, headers=headers, json=payload)

print(response.text)

**Explanation**

Vision Alignment enables you to customize a Vision Language Model (VLM) using your own image dataset, allowing the model to better understand and respond to domain-specific visual content. Instead of relying solely on the model's general knowledge, alignment adapts the model to your organization's data and use case.

The Vision Alignment workflow in this cookbook consists of the following steps:

1. Upload a vision dataset in JSONL format.
2. Generate benchmark questions using **15% of the uploaded dataset** to evaluate the aligned model.
3. Use the remaining **85% of the dataset** for alignment training.
4. Create an alignment project by selecting an alignment-ready Vision Language Model.
5. Monitor the training progress until the alignment is complete.
6. Perform inference using the newly aligned model.

By following this workflow, you can create a vision model that produces more accurate and context-aware responses for your specific application.

**List of available Vision Model**

| Model ID | Model Name |
|----------|------------|
| `qwen2-vl-2b-instruct` | Qwen2-VL-2B-Instruct |
| `qwen2p5-vl-32b-instruct` | Qwen2.5-VL-32B-Instruct |
| `qwen3-vl-30b-a3b-thinking` | Qwen3-VL-30B-A3B-Thinking |
| `qwen3-vl-235b-a22b-instruct` | Qwen3-VL-235B-A22B-Instruct |

**Conclusion**

In this cookbook, you learned how to build a complete Vision Alignment pipeline using the Nugen API. Starting with a vision dataset, you uploaded the data, generated benchmark questions-answers, created an alignment project, monitored the alignment process, and finally used the aligned model for inference.

The benchmark generated from **15% of the uploaded dataset** helps evaluate the quality of the aligned model, while the remaining **85%** is used to improve the model during training. This workflow provides a simple and effective way to adapt a Vision Language Model to domain-specific image understanding tasks.